In [1]:
import re
import glob
import zipfile
import time
from pathlib import Path
import json
import requests
import pandas as pd
from urllib.request import urlretrieve

In [2]:
# Measurement helper
import time, psutil, os, functools

_proc = psutil.Process(os.getpid())

def measure(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        m0 = _proc.memory_info().rss / 1e6
        c0 = time.process_time()
        t0 = time.perf_counter()
        out = fn(*args, **kwargs)
        print(f"{fn.__name__}:  wall={time.perf_counter()-t0:.1f}s  "
              f"cpu={time.process_time()-c0:.1f}s  "
              f"mem Δ{_proc.memory_info().rss/1e6 - m0:+.0f} MB")
        return out
    return wrapper

## 1. Downloading the Data

In [3]:
repo_root_dir = Path.cwd().parent
output_dir = repo_root_dir / "data" / "rainfall_over_nsw"
output_dir.mkdir(parents=True, exist_ok=True)
FORCE_DOWNLOAD = False 

In [4]:
# Necessary metadata
# Dataset used: https://figshare.com/articles/dataset/Daily_rainfall_over_NSW_Australia/14096681
article_id = 14096681  # this is the unique identifier of the article on figshare
url = f"https://api.figshare.com/v2/articles/{article_id}"
headers = {"Content-Type": "application/json"}

In [5]:
response = requests.request("GET", url, headers=headers)
response.raise_for_status() # check if the request was successful
data = json.loads(response.text)  # this contains all the articles data, feel free to check it out
files = data["files"]             # this is just the data about the files, which is what we want
files

[{'id': 26579150,
  'name': 'daily_rainfall_2014.png',
  'size': 58863,
  'is_link_only': False,
  'download_url': 'https://ndownloader.figshare.com/files/26579150',
  'supplied_md5': 'fd32a2ffde300a31f8d63b1825d47e5e',
  'computed_md5': 'fd32a2ffde300a31f8d63b1825d47e5e',
  'mimetype': 'image/png'},
 {'id': 26579171,
  'name': 'environment.yml',
  'size': 192,
  'is_link_only': False,
  'download_url': 'https://ndownloader.figshare.com/files/26579171',
  'supplied_md5': '060b2020017eed93a1ee7dd8c65b2f34',
  'computed_md5': '060b2020017eed93a1ee7dd8c65b2f34',
  'mimetype': 'text/plain'},
 {'id': 26586554,
  'name': 'README.md',
  'size': 5422,
  'is_link_only': False,
  'download_url': 'https://ndownloader.figshare.com/files/26586554',
  'supplied_md5': '61858c6cc0e6a6d6663a7e4c75bbd88c',
  'computed_md5': '61858c6cc0e6a6d6663a7e4c75bbd88c',
  'mimetype': 'text/x-python'},
 {'id': 26766812,
  'name': 'data.zip',
  'size': 814041183,
  'is_link_only': False,
  'download_url': 'https://n

In [ ]:
files_to_dl = ["data.zip"] 
for file in files:
    if file["name"] in files_to_dl:

        output_path = output_dir / file["name"]

        if output_path.exists() and not FORCE_DOWNLOAD:
            print(f"{file['name']} already exists. Skipping.")
        else:
            output_path.parent.mkdir(exist_ok=True)  # ensure folder exists

            print(f"Downloading {file['name']}...")
            urlretrieve(file["download_url"], output_path)

In [8]:
zip_path = output_dir / "data.zip"
extract_dir = output_dir / "data"

if not extract_dir.exists():
    print("Extracting files...")
    with zipfile.ZipFile(zip_path, "r") as f:
        f.extractall(output_dir)
else:
    print("Files already extracted. Skipping extraction.")

Extracting files...


## 2. Combining data CSVs

- exclude file `observed_daily_rainfall_SYD.csv` for this current part of the project
- Create new column based on model name, taken from the file name
    - EX: `SAM0-UNICON_daily_rainfall_NSW.csv` will have a value "SAM0-UNICON" under the new `model` column
- Existing columns: time,lat_min,lat_max,lon_min,lon_max,rain (mm/day)


In [4]:
### just listing to get an idea how individual file looks like 
use_cols = ["time", "lat_min", "lat_max", "lon_min","lon_max","rain (mm/day)"]
df = pd.read_csv("../data/rainfall_over_nsw/SAM0-UNICON_daily_rainfall_NSW.csv", usecols=use_cols)
df

,time,lat_min,lat_max,lon_min,lon_max,rain (mm/day)
0,1889-01-01 12:00:00,-35.811518,-34.86911,140.625,141.875,3.045650e-13
1,1889-01-02 12:00:00,-35.811518,-34.86911,140.625,141.875,3.572392e-04
2,1889-01-03 12:00:00,-35.811518,-34.86911,140.625,141.875,9.431562e-03
3,1889-01-04 12:00:00,-35.811518,-34.86911,140.625,141.875,9.663865e-02
4,1889-01-05 12:00:00,-35.811518,-34.86911,140.625,141.875,0.000000e+00
...,...,...,...,...,...,...
3541148,2014-12-27 12:00:00,-30.157068,-29.21466,153.125,154.375,6.689683e+00
3541149,2014-12-28 12:00:00,-30.157068,-29.21466,153.125,154.375,7.862555e+00
3541150,2014-12-29 12:00:00,-30.157068,-29.21466,153.125,154.375,1.000503e+01
3541151,2014-12-30 12:00:00,-30.157068,-29.21466,153.125,154.375,8.541592e+00


In [7]:
# Task 2 combine CSVs + add `model`, measured with @measure
output_path = Path(output_dir / "combined_data.csv")
FORCE_MERGE = False

@measure
def combine_rainfall_csvs(output_path: Path, force_merge: bool = False) -> pd.DataFrame:
    source_dirs = [Path(output_dir), Path(output_dir) / "data"]  # handle either layout
    csv_paths: list[Path] = []
    for d in source_dirs:
        if d.exists():
            csv_paths.extend(sorted(d.glob("*.csv")))

    csv_paths = [p for p in csv_paths if p.name != "observed_daily_rainfall_SYD.csv"]

    if not csv_paths:
        raise FileNotFoundError(
            f"No CSVs found in {[str(d) for d in source_dirs]} (after excluding observed)."
        )

    if output_path.exists() and not force_merge:
        print(f"Merged file already exists at {output_path.name}. Skipping.")
        return pd.read_csv(output_path)

    dfs: list[pd.DataFrame] = []
    for p in csv_paths:
        # Model name is the part of the filename before the first underscore
        m = re.search(r"([^_]*)", p.name)
        model = m.group(1) if m else ""

        dfi = pd.read_csv(p, usecols=use_cols)
        dfi.insert(len(dfi.columns), "model", model)  # ensure last column
        dfs.append(dfi)

    combined = pd.concat(dfs, ignore_index=True)
    combined.to_csv(output_path, index=False)
    print(f"Wrote {len(combined):,} rows to {output_path.name}")
    return combined

combined = combine_rainfall_csvs(output_path=output_path, force_merge=FORCE_MERGE)
combined.head()

Wrote 62,467,843 rows to combined_data.csv
combine_rainfall_csvs:  wall=270.0s  cpu=151.7s  mem Δ-840 MB


,time,lat_min,lat_max,lon_min,lon_max,rain (mm/day),model
0,1889-01-01 12:00:00,-36.25,-35.0,140.625,142.5,3.293256e-13,ACCESS-CM2
1,1889-01-02 12:00:00,-36.25,-35.0,140.625,142.5,0.000000e+00,ACCESS-CM2
2,1889-01-03 12:00:00,-36.25,-35.0,140.625,142.5,0.000000e+00,ACCESS-CM2
3,1889-01-04 12:00:00,-36.25,-35.0,140.625,142.5,0.000000e+00,ACCESS-CM2
4,1889-01-05 12:00:00,-36.25,-35.0,140.625,142.5,1.047658e-02,ACCESS-CM2


### Task 2 (step 4): compare + reflect

- Record the `@measure` output (wall / cpu / mem Δ) for the combine step on your machine and at least 1–3 classmates’ machines.
- In 3–5 sentences, reflect on what you observe. In particular, comment on the **wall/cpu ratio** (I/O-bound vs CPU-bound) and what might explain differences.

ME: combine_rainfall_csvs:  wall=270.0s  cpu=151.7s  mem Δ-840 MB

